# Module 04, Unit 02 — The Hessian & Convexity

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 04 | Unit 02

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridge** | Ridge regression closed-form solution; Newton's method vs gradient descent `[GLM]` |
| **Prerequisite** | Module 04, Unit 01 — Critical Points & the Second-Derivative Test |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] Define positive definite, negative definite, and indefinite matrices and test for each using eigenvalues
- [ ] State the definition of a convex function and the first- and second-order characterisations
- [ ] Explain why a convex loss function has no saddle points and its unique critical point is the global minimum
- [ ] Derive the Ridge regression closed-form solution by setting $\nabla J = \boldsymbol{0}$ and solving
- [ ] Write the Newton's method update rule and explain the role of $\boldsymbol{H}^{-1}$ as a curvature correction
- [ ] Compare gradient descent and Newton's method on a convex quadratic and describe when each is preferred

---

## Why This Unit

Unit 01 gave us tools to classify individual critical points. This unit steps back and asks: is there a global property of the loss function that guarantees we only have one critical point, and that it is the global minimum?

The answer is **convexity**. Convex loss functions are the foundation of classical statistics — OLS, Ridge, Lasso, logistic regression, and the Gaussian MLE are all convex. For convex problems, gradient descent always converges to the right answer, the Hessian has a definite sign everywhere, and the critical point equation $\nabla f = \boldsymbol{0}$ has a unique solution.

Understanding convexity also unlocks Newton's method — a second-order optimizer that uses the Hessian to take geometrically correct steps, converging quadratically where gradient descent converges only linearly.

---

## 1. Positive Definiteness

A symmetric matrix $\boldsymbol{A} \in \mathbb{R}^{n \times n}$ is classified by the sign of the quadratic form $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h}$:

| Name | Condition | Eigenvalues | Interpretation |
|---|---|---|---|
| **Positive definite** (PD) | $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h} > 0$ for all $\boldsymbol{h} \neq \boldsymbol{0}$ | All $\lambda_i > 0$ | Hessian at a local min |
| **Positive semi-definite** (PSD) | $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h} \geq 0$ for all $\boldsymbol{h}$ | All $\lambda_i \geq 0$ | Boundary case |
| **Negative definite** (ND) | $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h} < 0$ for all $\boldsymbol{h} \neq \boldsymbol{0}$ | All $\lambda_i < 0$ | Hessian at a local max |
| **Indefinite** | Sign of $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h}$ depends on $\boldsymbol{h}$ | Mixed signs | Saddle point |

**Why eigenvalues determine definiteness.** Since $\boldsymbol{A}$ is symmetric, it has an orthonormal eigenbasis $\{\boldsymbol{q}_1, \ldots, \boldsymbol{q}_n\}$ with real eigenvalues $\lambda_1, \ldots, \lambda_n$. Any $\boldsymbol{h}$ can be written as $\boldsymbol{h} = \sum_i c_i \boldsymbol{q}_i$, giving:

$$
\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h} = \sum_{i=1}^n \lambda_i c_i^2
$$

This is a weighted sum of squares with weights $\lambda_i$. It is positive for all $\boldsymbol{h} \neq \boldsymbol{0}$ if and only if all $\lambda_i > 0$.

**Sylvester's criterion** (practical test for PD without computing eigenvalues): $\boldsymbol{A}$ is positive definite if and only if all **leading principal minors** are positive — i.e., $A_{11} > 0$, $\det\begin{pmatrix}A_{11}&A_{12}\\A_{21}&A_{22}\end{pmatrix} > 0$, and so on for all $k \times k$ leading submatrices.

**Worked Example 1.1.** Is $\boldsymbol{A} = \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix}$ positive definite?

Leading minors: $3 > 0$ and $\det(\boldsymbol{A}) = 6 - 1 = 5 > 0$. Yes, $\boldsymbol{A}$ is PD.

Eigenvalues: $\lambda = \frac{5 \pm \sqrt{25 - 20}}{2} = \frac{5 \pm \sqrt{5}}{2}$. Both positive. ✓

---

## 2. Convex Functions

**Definition (geometric).** A function $f : \mathbb{R}^n \to \mathbb{R}$ is **convex** if for all $\boldsymbol{x}, \boldsymbol{y} \in \mathbb{R}^n$ and all $t \in [0, 1]$:

$$
f(t\boldsymbol{x} + (1-t)\boldsymbol{y}) \leq t\,f(\boldsymbol{x}) + (1-t)\,f(\boldsymbol{y})
$$

The function value at any convex combination of two points lies at or below the chord connecting them. Geometrically: the graph of $f$ lies below every secant line.

$f$ is **strictly convex** if the inequality is strict for $\boldsymbol{x} \neq \boldsymbol{y}$ and $t \in (0,1)$.

### 2.1 First-Order Characterisation

For differentiable $f$, convexity is equivalent to: the tangent hyperplane at any point lies entirely **below** (or on) the graph:

$$
f(\boldsymbol{y}) \geq f(\boldsymbol{x}) + \nabla f(\boldsymbol{x})^{\top}(\boldsymbol{y} - \boldsymbol{x}) \qquad \text{for all } \boldsymbol{x}, \boldsymbol{y}
$$

**Consequence for optimization**: if $\nabla f(\boldsymbol{x}^*) = \boldsymbol{0}$ and $f$ is convex, then for all $\boldsymbol{y}$:

$$
f(\boldsymbol{y}) \geq f(\boldsymbol{x}^*) + \boldsymbol{0}^{\top}(\boldsymbol{y} - \boldsymbol{x}^*) = f(\boldsymbol{x}^*)
$$

So any critical point of a convex function is the **global minimum**. There is no need to check whether it is a local minimum or saddle point — for convex functions, the only critical point is the global minimum.

### 2.2 Second-Order Characterisation

For twice-differentiable $f$:

$$
f \text{ is convex} \iff \boldsymbol{H}f(\boldsymbol{x}) \text{ is positive semi-definite for all } \boldsymbol{x}
$$

$$
f \text{ is strictly convex} \impliedby \boldsymbol{H}f(\boldsymbol{x}) \text{ is positive definite for all } \boldsymbol{x}
$$

**Intuition**: the Hessian measures the curvature of $f$. Positive curvature everywhere means the function curves upward in every direction from every point — the defining property of a bowl shape.

### 2.3 Convexity-Preserving Operations

These rules let you verify convexity of complex functions by decomposition:

- **Non-negative linear combination**: if $f$, $g$ convex and $\alpha, \beta \geq 0$, then $\alpha f + \beta g$ is convex
- **Composition with affine map**: $f(\boldsymbol{A}\boldsymbol{x} + \boldsymbol{b})$ is convex if $f$ is convex
- **Pointwise maximum**: $\max(f, g)$ is convex if $f$, $g$ convex
- **Log-sum-exp**: $\log\sum_i e^{x_i}$ is convex (important in softmax / logistic regression)

---

## 3. Newton's Method

Gradient descent moves in the direction $-\nabla f$ with a fixed step size. This ignores curvature — it treats a steep narrow valley the same as a gentle broad bowl. **Newton's method** corrects for curvature by using the Hessian.

### 3.1 Derivation

At the current point $\boldsymbol{\theta}^{(t)}$, approximate $f$ by its second-order Taylor expansion:

$$
f(\boldsymbol{\theta}) \approx f(\boldsymbol{\theta}^{(t)}) + \nabla f(\boldsymbol{\theta}^{(t)})^{\top}(\boldsymbol{\theta} - \boldsymbol{\theta}^{(t)}) + \frac{1}{2}(\boldsymbol{\theta} - \boldsymbol{\theta}^{(t)})^{\top}\boldsymbol{H}f(\boldsymbol{\theta}^{(t)})(\boldsymbol{\theta} - \boldsymbol{\theta}^{(t)})
$$

Minimize this quadratic approximation over $\boldsymbol{\theta}$ by differentiating and setting to zero:

$$
\nabla f(\boldsymbol{\theta}^{(t)}) + \boldsymbol{H}f(\boldsymbol{\theta}^{(t)})(\boldsymbol{\theta} - \boldsymbol{\theta}^{(t)}) = \boldsymbol{0}
$$

Solving for the next iterate (assuming $\boldsymbol{H}f$ is invertible):

$$
\boxed{\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \left[\boldsymbol{H}f(\boldsymbol{\theta}^{(t)})\right]^{-1} \nabla f(\boldsymbol{\theta}^{(t)})}
$$

Compare to gradient descent: $\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \alpha\,\nabla f(\boldsymbol{\theta}^{(t)})$.

Newton's method replaces the scalar learning rate $\alpha$ with the matrix $[\boldsymbol{H}f]^{-1}$ — a curvature-adapted step. In directions where $f$ is highly curved (large eigenvalues of $\boldsymbol{H}$), the step is small. In flat directions (small eigenvalues), the step is large. This is exactly the right behavior.

### 3.2 Convergence

| Method | Convergence rate | Steps to reach $\varepsilon$ precision | Cost per step |
|---|---|---|---|
| Gradient descent | Linear: error $\propto \rho^t$ | $O(\log(1/\varepsilon))$ | $O(n)$ |
| Newton's method | Quadratic: error $\propto (\text{error}_{t})^2$ | $O(\log\log(1/\varepsilon))$ | $O(n^3)$ (Hessian inversion) |

Newton's method converges much faster per iteration, but each step is expensive: computing and inverting the $n \times n$ Hessian costs $O(n^3)$. For large $n$ (millions of neural network parameters), this is prohibitive. Practical optimizers like L-BFGS approximate $\boldsymbol{H}^{-1}$ without ever forming it explicitly.

### 3.3 On a Convex Quadratic, Newton Converges in One Step

For $f(\boldsymbol{\theta}) = \frac{1}{2}\boldsymbol{\theta}^{\top}\boldsymbol{A}\boldsymbol{\theta} - \boldsymbol{b}^{\top}\boldsymbol{\theta}$ with $\boldsymbol{A}$ PD, the Hessian is constant $\boldsymbol{H}f = \boldsymbol{A}$ and Newton gives:

$$
\boldsymbol{\theta}^{(1)} = \boldsymbol{\theta}^{(0)} - \boldsymbol{A}^{-1}(\boldsymbol{A}\boldsymbol{\theta}^{(0)} - \boldsymbol{b}) = \boldsymbol{A}^{-1}\boldsymbol{b}
$$

One step, regardless of starting point. Gradient descent on the same problem requires many iterations, especially when $\boldsymbol{A}$ has a large condition number.

---

## 4. Statistical Bridge — Ridge Regression Closed Form `[GLM]`

Ridge regression was introduced in Module 00 Unit 01 as a penalty on $\|\boldsymbol{\beta}\|^2$. We can now derive its solution exactly — it is a convex optimization problem with a closed-form answer.

### 4.1 The Objective

Given design matrix $\boldsymbol{X} \in \mathbb{R}^{n \times p}$ and response $\boldsymbol{y} \in \mathbb{R}^n$, Ridge regression minimizes:

$$
J(\boldsymbol{\beta}) = \|\boldsymbol{y} - \boldsymbol{X}\boldsymbol{\beta}\|^2 + \lambda\|\boldsymbol{\beta}\|^2
$$

Expanding:

$$
J(\boldsymbol{\beta}) = \boldsymbol{y}^{\top}\boldsymbol{y} - 2\boldsymbol{y}^{\top}\boldsymbol{X}\boldsymbol{\beta} + \boldsymbol{\beta}^{\top}\boldsymbol{X}^{\top}\boldsymbol{X}\boldsymbol{\beta} + \lambda\boldsymbol{\beta}^{\top}\boldsymbol{\beta}
$$

### 4.2 Gradient

Using the gradient reference table from Module 03 Unit 01:

$$
\nabla_{\boldsymbol{\beta}}\,J = -2\boldsymbol{X}^{\top}\boldsymbol{y} + 2\boldsymbol{X}^{\top}\boldsymbol{X}\boldsymbol{\beta} + 2\lambda\boldsymbol{\beta} = 2\left[(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})\boldsymbol{\beta} - \boldsymbol{X}^{\top}\boldsymbol{y}\right]
$$

### 4.3 Solving $\nabla J = \boldsymbol{0}$

$$
(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})\hat{\boldsymbol{\beta}} = \boldsymbol{X}^{\top}\boldsymbol{y}
$$

$$
\boxed{\hat{\boldsymbol{\beta}}_{\text{Ridge}} = (\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y}}
$$

### 4.4 Why the Inverse Exists

The Hessian of $J$ is:

$$
\boldsymbol{H}J = 2(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})
$$

$\boldsymbol{X}^{\top}\boldsymbol{X}$ is always positive semi-definite (its eigenvalues are $\geq 0$). Adding $\lambda\boldsymbol{I}$ with $\lambda > 0$ shifts all eigenvalues up by $\lambda$, making them strictly positive. Therefore:

- $\boldsymbol{H}J$ is **positive definite** for any $\lambda > 0$
- $J$ is **strictly convex** — unique global minimum
- $(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})$ is **invertible** — the solution always exists

This is the regularization doing double duty: it not only controls model complexity, it also guarantees the optimization problem is well-posed.

For comparison, OLS ($\lambda = 0$) gives $\hat{\boldsymbol{\beta}}_{\text{OLS}} = (\boldsymbol{X}^{\top}\boldsymbol{X})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y}$, which requires $\boldsymbol{X}^{\top}\boldsymbol{X}$ to be invertible — a condition that fails when $p > n$ or when features are collinear. Ridge avoids this by adding $\lambda\boldsymbol{I}$.

### 4.5 Newton's Method Recovers the Ridge Solution in One Step

Since $J$ is a convex quadratic, Newton's method with $\boldsymbol{\beta}^{(0)} = \boldsymbol{0}$ gives:

$$
\boldsymbol{\beta}^{(1)} = \boldsymbol{0} - [\boldsymbol{H}J]^{-1}\nabla J(\boldsymbol{0}) = -\frac{1}{2}(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})^{-1}(-2\boldsymbol{X}^{\top}\boldsymbol{y}) = (\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y}
$$

One Newton step from the origin reaches the exact Ridge solution.

---

## Python: Visualization & Verification

The cells below demonstrate and visualize the concepts above. **Run them in order.**

To run this notebook interactively, click the **rocket icon** at the top of the page and select **Open in Colab**.

In [ ]:
# ============================================================
# Install required libraries
# ============================================================
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy"]:
    install(pkg)

print("All packages installed.")

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sympy as sp

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 2,
    'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')

### Section 1 — Convexity: Geometric Verification

Verify the chord condition $f(t\boldsymbol{x} + (1-t)\boldsymbol{y}) \leq tf(\boldsymbol{x}) + (1-t)f(\boldsymbol{y})$ for a convex and a non-convex function, and visualize the Hessian eigenvalues along a path to confirm positive definiteness.

In [ ]:
# ============================================================
# Section 1 — Convexity geometric verification
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Convex: f(x,y) = x² + 2y² + xy ----
def f_convex(v):
    x, y = v
    return x**2 + 2*y**2 + x*y

# ---- Non-convex: f(x,y) = x² - 2y² + xy (indefinite Hessian) ----
def f_nonconvex(v):
    x, y = v
    return x**2 - 2*y**2 + x*y

p1 = np.array([-2.0, 1.5])
p2 = np.array([2.0, -1.0])
t_vals = np.linspace(0, 1, 200)

for ax, f_fn, title, color in [
    (axes[0], f_convex,    r'Convex: $f = x^2 + 2y^2 + xy$', TS_BLUE),
    (axes[1], f_nonconvex, r'Non-convex: $f = x^2 - 2y^2 + xy$', TS_RED),
]:
    f_on_line  = np.array([f_fn(t*p2 + (1-t)*p1) for t in t_vals])
    chord_vals = t_vals * f_fn(p2) + (1 - t_vals) * f_fn(p1)

    ax.plot(t_vals, f_on_line, color=color, lw=2.5, label='$f(tp_2 + (1-t)p_1)$')
    ax.plot(t_vals, chord_vals, color=TS_AMBER, lw=2.0, ls='--',
            label='Chord $tf(p_2) + (1-t)f(p_1)$')
    ax.fill_between(t_vals, f_on_line, chord_vals,
                    alpha=0.15, color=TS_GREEN if color == TS_BLUE else TS_RED)
    ax.plot([0, 1], [f_fn(p1), f_fn(p2)], 'o', color=TS_AMBER, markersize=8)
    ax.set_xlabel('t'); ax.set_ylabel('f value')
    ax.set_title(title)
    ax.legend(fontsize=9)

    # Annotate gap
    gap = chord_vals - f_on_line
    ax.text(0.5, (f_on_line[100] + chord_vals[100])/2,
            'chord ≥ f ✓' if gap.min() >= -1e-10 else 'chord < f ✗',
            ha='center', fontsize=10,
            color=TS_GREEN if gap.min() >= -1e-10 else TS_RED)

plt.suptitle('Chord condition — convex (left) vs non-convex (right)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# --- Hessian eigenvalues along the line segment ---
def hessian_convex(v):
    return np.array([[2.0, 1.0], [1.0, 4.0]])  # constant for this quadratic

def hessian_nonconvex(v):
    return np.array([[2.0, 1.0], [1.0, -4.0]])

print('Eigenvalues of H_convex    :', np.round(np.linalg.eigvalsh(hessian_convex(None)), 4))
print('All positive (PD)?', np.all(np.linalg.eigvalsh(hessian_convex(None)) > 0))
print('\nEigenvalues of H_nonconvex :', np.round(np.linalg.eigvalsh(hessian_nonconvex(None)), 4))
print('Mixed sign (indefinite)?', not np.all(np.linalg.eigvalsh(hessian_nonconvex(None)) > 0))

**What do you see?** The convex function (left) always lies below the chord — the green shaded region shows the gap is non-negative. The non-convex function (right) dips below the chord — the chord condition is violated, confirming non-convexity. The Hessian eigenvalue printout confirms the correspondence: all-positive eigenvalues for the convex function, mixed signs for the non-convex one.

### Section 2 — Gradient Descent vs Newton's Method

Run both optimizers on an ill-conditioned convex quadratic and compare convergence. The elongated contours make gradient descent slow while Newton converges in one step.

In [ ]:
# ============================================================
# Section 2 — Gradient descent vs Newton's method
# ============================================================
# Convex quadratic: J(θ) = θᵀAθ/2 - bᵀθ
# A = [[10, 0],[0, 0.5]]  (condition number = 20)
A = np.array([[10.0, 0.0], [0.0, 0.5]])
b = np.array([1.0, 2.0])
theta_star = np.linalg.solve(A, b)   # exact solution

def J_val(theta):    return 0.5 * theta @ A @ theta - b @ theta
def grad_J(theta):   return A @ theta - b
def hess_J(theta):   return A   # constant

# --- Gradient descent ---
lr = 0.09   # largest stable lr ≈ 2/(λ_max+λ_min)
theta = np.array([-1.5, -1.5])
gd_path = [theta.copy()]
gd_loss = [J_val(theta)]
for _ in range(60):
    theta = theta - lr * grad_J(theta)
    gd_path.append(theta.copy()); gd_loss.append(J_val(theta))
gd_path = np.array(gd_path)

# --- Newton's method ---
theta_n = np.array([-1.5, -1.5])
nm_path = [theta_n.copy()]
nm_loss = [J_val(theta_n)]
for _ in range(5):
    H_inv = np.linalg.inv(hess_J(theta_n))
    theta_n = theta_n - H_inv @ grad_J(theta_n)
    nm_path.append(theta_n.copy()); nm_loss.append(J_val(theta_n))
    if np.linalg.norm(grad_J(theta_n)) < 1e-12:
        break
nm_path = np.array(nm_path)

# --- Contour grid ---
t1g = np.linspace(-2.2, 1.2, 300)
t2g = np.linspace(-2.2, 4.5, 300)
T1, T2 = np.meshgrid(t1g, t2g)
ZZ = 0.5*(A[0,0]*T1**2 + A[1,1]*T2**2) - b[0]*T1 - b[1]*T2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, path, label, color, note in [
    (axes[0], gd_path, f'Gradient descent (α={lr})', TS_BLUE,
     f'{len(gd_path)-1} steps'),
    (axes[1], nm_path, "Newton's method", TS_GREEN,
     f'{len(nm_path)-1} step(s)'),
]:
    ax.contourf(T1, T2, ZZ, levels=np.logspace(-1, 1.5, 22),
                cmap='Blues', alpha=0.20)
    ax.contour(T1, T2, ZZ,  levels=np.logspace(-1, 1.5, 22),
               colors=[TS_GRAY], alpha=0.45, linewidths=0.8)
    ax.plot(path[:,0], path[:,1], 'o-', color=color,
            markersize=4, lw=1.8, alpha=0.9, label=label)
    ax.plot(*path[0], 's', color=TS_GRAY, markersize=9, zorder=6, label='Start')
    ax.plot(*theta_star, '*', color=TS_AMBER, markersize=14, zorder=6, label='Optimum')
    ax.set_xlabel(r'$\theta_1$'); ax.set_ylabel(r'$\theta_2$')
    ax.set_title(f'{label}\n{note}, final loss = {J_val(path[-1]):.2e}')
    ax.legend(fontsize=8)

plt.suptitle(r'$J(\theta) = \frac{1}{2}\theta^\top A\theta - b^\top\theta$,  condition number = 20',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f'Exact solution θ*         : {theta_star}')
print(f'Gradient descent final θ  : {np.round(gd_path[-1], 5)}')
print(f"Newton's method final θ   : {np.round(nm_path[-1], 12)}")
print(f"\nNewton error from θ*      : {np.linalg.norm(nm_path[-1] - theta_star):.2e}")

**What do you see?**

- **Gradient descent**: zigzags along the elongated contours. The large condition number ($\lambda_{\max}/\lambda_{\min} = 20$) forces slow convergence — each step overshoots in the $\theta_2$ direction while barely moving in $\theta_1$.
- **Newton's method**: jumps directly to the optimum in one step. Because the loss is an exact quadratic with constant Hessian, the second-order Taylor approximation is exact and Newton converges immediately.
- This comparison quantifies the cost of ignoring curvature. In practice, neural network loss surfaces are not quadratic, but near a minimum they are approximately quadratic — which is why quasi-Newton methods like L-BFGS converge much faster than gradient descent in the final stage of training.

**Try modifying**: Set `A = np.array([[10.0, 0.0],[0.0, 9.5]])` (condition number ≈ 1.05, nearly spherical). Does gradient descent converge as fast as Newton now?

### Section 3 — Ridge Regression: Closed Form, Gradient Descent, and Shrinkage

Verify the Ridge closed-form solution, show gradient descent converging to the same answer, and visualize how $\lambda$ controls shrinkage of the coefficients.

In [ ]:
# ============================================================
# Section 3 — Ridge regression: closed form and shrinkage
# ============================================================
rng = np.random.default_rng(7)
n, p = 60, 5
X = rng.standard_normal((n, p))
beta_true = np.array([3.0, -2.0, 1.5, 0.5, -0.3])
y = X @ beta_true + rng.standard_normal(n) * 0.8

# --- Closed-form Ridge solution ---
def ridge_closed(X, y, lam):
    """(XᵀX + λI)⁻¹ Xᵀy"""
    p = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)

lam = 2.0
beta_ridge = ridge_closed(X, y, lam)

# --- Gradient descent on Ridge objective ---
def ridge_loss(beta, X, y, lam):
    r = y - X @ beta
    return r @ r + lam * beta @ beta

def ridge_grad(beta, X, y, lam):
    return -2 * X.T @ (y - X @ beta) + 2 * lam * beta

beta_gd = np.zeros(p)
lr_gd   = 1.0 / (2 * (np.linalg.norm(X.T @ X, ord=2) + lam))
loss_hist = [ridge_loss(beta_gd, X, y, lam)]
for _ in range(500):
    beta_gd = beta_gd - lr_gd * ridge_grad(beta_gd, X, y, lam)
    loss_hist.append(ridge_loss(beta_gd, X, y, lam))

print('Ridge closed-form β̂ :', np.round(beta_ridge, 4))
print('Ridge GD final β̂    :', np.round(beta_gd, 4))
print(f'Max difference        : {np.abs(beta_ridge - beta_gd).max():.2e}  ✓')

# --- Shrinkage path ---
lambdas = np.logspace(-2, 3, 120)
beta_paths = np.array([ridge_closed(X, y, l) for l in lambdas])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: coefficient shrinkage path
ax = axes[0]
colors_coef = [TS_BLUE, TS_AMBER, TS_GREEN, TS_RED, TS_GRAY]
for j, col in enumerate(colors_coef):
    ax.semilogx(lambdas, beta_paths[:, j], color=col, lw=2,
                label=fr'$\beta_{j+1}$ (true={beta_true[j]})')
ax.axvline(lam, color='black', lw=1.2, ls='--', alpha=0.6,
           label=f'λ = {lam}')
ax.axhline(0, color=TS_GRAY, lw=0.8)
ax.set_xlabel(r'$\lambda$ (log scale)')
ax.set_ylabel(r'$\hat{\beta}_j$')
ax.set_title('Ridge shrinkage path')
ax.legend(fontsize=8, loc='upper right')

# Right: gradient descent loss curve
ax2 = axes[1]
ax2.semilogy(loss_hist, color=TS_BLUE, lw=2)
ax2.axhline(ridge_loss(beta_ridge, X, y, lam), color=TS_AMBER,
            lw=1.5, ls='--', label='Closed-form optimum')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Ridge loss (log scale)')
ax2.set_title('Gradient descent converges to closed-form solution')
ax2.legend(fontsize=9)

plt.suptitle(r'Ridge regression: $J(\beta) = \|y - X\beta\|^2 + \lambda\|\beta\|^2$',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?**

- **Closed form and gradient descent agree** to better than $10^{-5}$ — the same objective, reached two different ways. The closed-form solution is exact; gradient descent converges to it asymptotically.
- **Shrinkage path**: as $\lambda \to 0$ the coefficients approach OLS estimates; as $\lambda \to \infty$ they are all shrunk toward zero. The rate of shrinkage differs by coefficient — larger coefficients are shrunk more in absolute terms. This is the bias-variance tradeoff made concrete: more regularization reduces variance at the cost of bias.
- **Why coefficients never reach exactly zero**: Ridge adds a spherical penalty $\lambda\|\boldsymbol{\beta}\|^2$. Its level curves are circles — they contact the OLS constraint set smoothly, not at corners. Lasso ($\lambda\|\boldsymbol{\beta}\|_1$) has diamond-shaped level curves with corners on the axes — this is why Lasso produces exact zeros and Ridge does not. We prove this in Unit 03.

In [ ]:
# Extension workspace
# Suggestions:
# 1. Verify Sylvester's criterion for a 3×3 matrix by checking all
#    leading minors and comparing to eigenvalue signs.
#
# 2. Implement Newton's method for logistic regression (from Module 03
#    Unit 02). Compute the Hessian H = (1/n)XᵀDX where D = diag(p(1-p)),
#    and run the Newton update. Compare convergence to gradient descent.
#
# 3. What is the condition number of (XᵀX + λI) as a function of λ?
#    Plot it. How does regularization improve the conditioning of the
#    linear system we need to solve?


---

## Summary

| Concept | Statement |
|---|---|
| Positive definite | All eigenvalues of $\boldsymbol{A}$ positive; $\boldsymbol{h}^{\top}\boldsymbol{A}\boldsymbol{h} > 0$ ∀ $\boldsymbol{h} \neq \boldsymbol{0}$ |
| Convexity (geometric) | $f(t\boldsymbol{x} + (1-t)\boldsymbol{y}) \leq tf(\boldsymbol{x}) + (1-t)f(\boldsymbol{y})$ |
| Convexity (second-order) | $\boldsymbol{H}f(\boldsymbol{x})$ PSD for all $\boldsymbol{x}$ |
| Global min guarantee | Convex $f$ + critical point $\implies$ global minimum |
| Newton update | $\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - [\boldsymbol{H}f]^{-1}\nabla f$ |
| Newton on quadratic | Converges in one step from any starting point |
| Ridge loss Hessian | $\boldsymbol{H}J = 2(\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})$ — PD for any $\lambda > 0$ |
| Ridge closed form | $\hat{\boldsymbol{\beta}} = (\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y}$ |

**Up next:** Unit 03 — Constrained Optimization & Lagrange Multipliers

We extend optimization to problems with equality and inequality constraints, derive the Lagrangian and KKT conditions, and apply them to find the maximum entropy distribution and understand Lasso as a constrained problem.

---
*Module 04, Unit 02 — Threat Surfaces | fischer³ Education*